### How `gas_ccs_capacity_factor` is calculated

In this project, `gas_ccs_capacity_factor` is **not** the traditional power-system definition:

\[
\text{capacity factor} = \frac{\text{annual generation}}{\text{installed capacity} \times \text{hours per year}}
\]

Instead, in the **No H2** model it is calculated as:

```python
gas_ccs_capacity_factor = (sim_df[gas_ccs_column] > 0).mean()


In [3]:
import pandas as pd

import src.assumptions as A
from src.units import Units as U
from src.demand_model import predicted_demand, DemandMode
from src.data import renewable_capacity_factors
from src.supply_model import daily_renewables_capacity
from src.power_system_no_h2 import PowerSystemNoH2

# Match the notebook setup
REN_GW = 200
DAC_GW = 2.6
GAS_GW = 99
ENABLE_IMPORTS = False
CAPACITY_FACTORS_SOURCE = "era5_2024"
DEMAND_MODE = DemandMode.SEASONAL

# Build daily demand and supply
demand_df = predicted_demand(mode=DEMAND_MODE, historical="era5", filter_ldz=True)

daily_cf = renewable_capacity_factors.get_renewable_capacity_factors(
    source=CAPACITY_FACTORS_SOURCE,
    resample="D",
)

supply_df = pd.DataFrame(
    {REN_GW: daily_renewables_capacity(REN_GW * U.GW, daily_cf)},
    index=daily_cf.index,
)

common_idx = supply_df.index.intersection(demand_df.index)
supply_df = supply_df.reindex(common_idx)
demand_df = demand_df.reindex(common_idx)

net_supply_df = supply_df.sub(demand_df["demand"], axis=0)
net_supply_df = net_supply_df.rename(columns={REN_GW: f"S-D(TWh),Ren={REN_GW}GW"})

# Run No H2 system
ps_no_h2 = PowerSystemNoH2(
    renewable_capacity=REN_GW * U.GW,
    dac_capacity=DAC_GW * U.GW,
    gas_ccs_capacity=GAS_GW * U.GW,
    enable_imports=ENABLE_IMPORTS,
    capacity_factors_source=CAPACITY_FACTORS_SOURCE,
)

sim_no_h2 = ps_no_h2.run_simulation(net_supply_df)
analysis_no_h2 = ps_no_h2.analyze_simulation_results(sim_no_h2)

print(f"gas_ccs_capacity_factor = {analysis_no_h2['gas_ccs_capacity_factor']:.4f}")
print(f"full value = {analysis_no_h2['gas_ccs_capacity_factor']}")


gas_ccs_capacity_factor = 0.3148
full value = 0.3148259504242667
